# Pipeline
This is where I build my pipeline.

## Idea
1. We will have two categories: Intersection and Address collisions (refer to `exploration.ipynb` for how I came to this conclusion).
2. We will separate the collisons dataset into two subsets, one for each category.

### Extract
1. Intersection and address collisions will have unrelated columns removed.
2. Intersection and address feature sets will also have unrelated columns removed, and will be transformed to contain only `id`, `description` and `geometry` columns.

### Transform
1. Intersection collisions will be mapped to the nearest intersection feature via spatial index, and string match score will be calculated via rapidfuzz
2. Address collisions will be mapped to the nearest address feature via spatial index, and string match score will be calculated via rapidfuzz

### Load
idk what happens now, I guess we just display the results.



In [92]:
# Importing necessary libraries

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape

address_pattern = r'^\d{1,5}\w?\s{0,2}\w+\s?\w+$'

# Function to fix multi point geoms
def to_point(geom):
    if geom.geom_type == "Point":
        return geom
    elif geom.geom_type == "MultiPoint":
        return list(geom.geoms)[0]  # take first point
    else:
        raise ValueError(f"Unsupported geometry type: {geom.geom_type}")


In [93]:
# Loading the collision data
collision_data = pd.read_csv("../data/raw/collisions.csv")

collision_data = collision_data[['collision_id', 'stname1', 'stname2', 'stname3', 'latitude', 'longitude']]

collision_data = gpd.GeoDataFrame(collision_data, geometry=gpd.points_from_xy(collision_data.longitude, collision_data.latitude), crs="EPSG:4326")
collision_data.to_csv("../data/processed/collisions.csv", index=False)
collision_data.count()

collision_id    20455
stname1         20455
stname2         18697
stname3          4659
latitude        20452
longitude       20452
geometry        20455
dtype: int64

In [94]:
# Loading and processing intersection data
from shapely.geometry import shape
import json

# Initialize the intersection data
intersection_data = pd.read_csv('../data/raw/intersections.csv')

# Rename columns for consistency
intersection_data = intersection_data.rename(columns={'INTERSECTION_ID': 'feature_id', 'INTERSECTION_DESC': 'feature_description'})

# Adding type column for use with address data
intersection_data['type'] = 'intersection'

# Extract only necessary columns
intersection_data = intersection_data[['feature_id', 'feature_description', 'type', 'geometry']]

# Filter out rows where description is null or empty
intersection_data = intersection_data[intersection_data['feature_description'].notna() & (intersection_data['feature_description'].str.strip() != '')]

# Convert the geometry from WKT to shapely geometry
intersection_data["geometry"] = intersection_data["geometry"].apply(lambda x: shape(json.loads(x)))
intersection_data = gpd.GeoDataFrame(intersection_data, geometry='geometry', crs="EPSG:4326")

intersection_data['geometry'] = intersection_data['geometry'].apply(lambda x: to_point(x))

intersection_data.to_csv("../data/processed/intersections.csv", index=False)

intersection_data.count()

feature_id             46257
feature_description    46257
type                   46257
geometry               46257
dtype: int64

In [95]:
# Loading and processing address data
from shapely.geometry import shape
import json

address_data = pd.read_csv('../data/raw/addresses.csv')

# Renaming columns for consistency
address_data = address_data.rename(columns={"ADDRESS_POINT_ID": "feature_id", "ADDRESS_FULL": "feature_description"})

# Adding type column for use with intersection data
address_data['type'] = 'address'

# Extracting necessary columns
address_data = address_data[["feature_id", "feature_description", 'type', "geometry"]]

# Filter out rows where description is null or empty
address_data = address_data[address_data['feature_description'].notna() & (address_data['feature_description'].str.strip() != '')]

# Constructing geometry 
address_data["geometry"] = address_data["geometry"].apply(lambda x: shape(json.loads(x)))

address_data = gpd.GeoDataFrame(address_data, geometry="geometry", crs="EPSG:4326")

# Cleaning up multipoint geometry
address_data["geometry"] = address_data["geometry"].apply(lambda x: to_point(x))

address_data.to_csv("../data/processed/addresses.csv")

address_data.count()

/tmp/ipykernel_26091/3179409233.py:5: DtypeWarning: Columns (0: HI_NUM_SUF) have mixed types. Specify dtype option on import or set low_memory=False.
  address_data = pd.read_csv('../data/raw/addresses.csv')


feature_id             525401
feature_description    525401
type                   525401
geometry               525401
dtype: int64

In [96]:
# Separate address collisions from the rest of the collisions

address_collisions = collision_data[collision_data['stname1'].str.match(address_pattern, na=False) | collision_data['stname2'].str.match(address_pattern, na=False)]

address_collisions.count()


collision_id    1286
stname1         1286
stname2          168
stname3          383
latitude        1286
longitude       1286
geometry        1286
dtype: int64

In [97]:
intersection_collisions = collision_data[~collision_data['collision_id'].isin(address_collisions['collision_id'])]

intersection_collisions.count()

collision_id    19169
stname1         19169
stname2         18529
stname3          4276
latitude        19166
longitude       19166
geometry        19169
dtype: int64

In [98]:
# Reproject collisions and features to common CRS

intersection_data = intersection_data.to_crs(epsg=32617)
intersection_collisions = intersection_collisions.to_crs(epsg=32617)

address_data = address_data.to_crs(epsg=32617)
address_collisions = address_collisions.to_crs(epsg=32617)

In [99]:
# Merge collision and feature dataset by distance using spatial index

# Intersections
intersection_merged = gpd.sjoin_nearest(intersection_collisions, intersection_data, distance_col="distance", how="left")

intersection_merged['stname2'] = intersection_merged['stname2'].fillna("")

intersection_merged['collision_description'] = (intersection_merged['stname1'] + " " + intersection_merged['stname2']).str.lower()

intersection_merged['feature_description'] = intersection_merged['feature_description'].str.lower().str.replace("/", "")

# Addresses
address_merged = gpd.sjoin_nearest(address_collisions, address_data, distance_col="distance")

# Filling null values with empty string to help with concat
address_merged['stname2'] = address_merged['stname2'].fillna("")

address_merged['collision_description'] = (address_merged['stname1'] + " " + address_merged["stname2"]).str.lower()

address_merged['feature_description'] = address_merged['feature_description'].str.lower()

intersection_merged.count()

collision_id             19921
stname1                  19921
stname2                  19921
stname3                   4505
latitude                 19918
longitude                19918
geometry                 19921
index_right              19918
feature_id               19918
feature_description      19918
type                     19918
distance                 19918
collision_description    19921
dtype: int64

In [100]:
# Compute string similarity scores

from rapidfuzz import fuzz

# Intersections
intersection_merged['match_score'] = [fuzz.token_sort_ratio(loc, desc) for loc, desc in zip(intersection_merged['collision_description'], intersection_merged['feature_description'])]

# Addresses
address_merged['match_score'] = [fuzz.token_sort_ratio(loc, desc) for loc, desc in zip(address_merged['collision_description'], address_merged['feature_description'])]

In [102]:
# sjoin_nearest creates duplicate rows if there are multiple nearest features for a given collision. We need to pick the feature with the highest string similarity score.

intersection_merged_sorted = intersection_merged.sort_values(by=['match_score'], ascending=False)

intersection_merged = intersection_merged_sorted.loc[~intersection_merged_sorted.index.duplicated(keep="first")]
intersection_merged.count()

address_merged_sorted = address_merged.sort_values(by=['match_score'], ascending=False)

address_merged = address_merged_sorted.loc[~address_merged_sorted.index.duplicated(keep="first")]

address_merged.count()

collision_id             1286
stname1                  1286
stname2                  1286
stname3                   383
latitude                 1286
longitude                1286
geometry                 1286
index_right              1286
feature_id               1286
feature_description      1286
type                     1286
distance                 1286
collision_description    1286
match_score              1286
dtype: int64

In [103]:
# Let's see some intersection collision stats!
print("Intersections\n", intersection_merged['match_score'].describe(), "\n")

print(intersection_merged['distance'].describe(), "\n")

Intersections
 count    19169.000000
mean        85.695790
std         19.838838
min          0.000000
25%         73.170732
50%        100.000000
75%        100.000000
max        100.000000
Name: match_score, dtype: float64 

count    19166.000000
mean        22.518454
std         34.630152
min          0.069788
25%          3.409349
50%          8.368245
75%         27.091333
max        411.611570
Name: distance, dtype: float64 



In [106]:
intersection_merged[intersection_merged['distance'].isna()]

,collision_id,stname1,stname2,stname3,latitude,longitude,geometry,index_right,feature_id,feature_description,type,distance,collision_description,match_score
7362,2011:1256815,THE QUEENSWAY,CULNAN AVE,NaN,NaN,NaN,POINT (NaN NaN),NaN,NaN,NaN,NaN,NaN,the queensway culnan ave,0.0
7363,2011:1256815,THE QUEENSWAY,CULNAN AVE,NaN,NaN,NaN,POINT (NaN NaN),NaN,NaN,NaN,NaN,NaN,the queensway culnan ave,0.0
7361,2011:1256815,THE QUEENSWAY,CULNAN AVE,NaN,NaN,NaN,POINT (NaN NaN),NaN,NaN,NaN,NaN,NaN,the queensway culnan ave,0.0


In [104]:
# Let's see some address collision stats!
print("Addresses\n", address_merged['match_score'].describe(), "\n")
print(address_merged['distance'].describe())


Addresses
 count    1286.000000
mean       81.272320
std        19.496454
min        19.354839
25%        76.470588
50%        87.500000
75%        93.333333
max       100.000000
Name: match_score, dtype: float64 

count    1286.000000
mean       36.923876
std        27.481132
min         4.460519
25%        21.708875
50%        27.647894
75%        42.699748
max       233.879434
Name: distance, dtype: float64


In [107]:
intersection_merged[(intersection_merged['match_score'] >= 70) & (intersection_merged['distance'] <= 30)][['distance', 'match_score']].describe()

,distance,match_score
count,12880.000000,12880.000000
mean,6.151876,95.114247
std,4.719778,9.047851
min,0.069788,70.000000
25%,2.603839,95.833333
50%,5.144978,100.000000
75%,8.646591,100.000000
max,29.894758,100.000000


In [109]:
address_merged[(address_merged['match_score'] >= 70) & (address_merged['distance'] <= 40)][['distance', 'match_score']].describe()

,distance,match_score
count,748.000000,748.000000
mean,24.266284,90.052312
std,6.835987,6.666076
min,5.961716,70.270270
25%,19.793079,86.666667
50%,23.612877,92.307692
75%,28.398806,93.750000
max,39.993820,100.000000
